In [ ]:

# Orphanet pipeline:
# - Download product1, product6, product4
# - Save XML files in the local data/ directory
# - Parse disease → gene (product6)
# - Save CSV in data/
# - Print first 3 rows
# ============================================================

import urllib.request
from pathlib import Path
from lxml import etree
import pandas as pd

# ----------------------------
# CONFIG
# ----------------------------
ORPHANET_BASE_URL = "https://www.orphadata.com/data/xml"

ORPHANET_PRODUCTS = {
    "product1": {
        "filename": "en_product1.xml",
        "url": f"{ORPHANET_BASE_URL}/en_product1.xml",
        "description": "Nomenclature and classification"
    },
    "product6": {  #  Disease → Gene
        "filename": "en_product6.xml",
        "url": f"{ORPHANET_BASE_URL}/en_product6.xml",
        "description": "Disease–gene associations"
    },
    "product4": {
        "filename": "en_product4.xml",
        "url": f"{ORPHANET_BASE_URL}/en_product4.xml",
        "description": "Phenotypes (HPO)"
    }
}

#  PATH RICHIESTO
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# DOWNLOAD FUNCTION
# ----------------------------
def download_orphanet_product(product_key):
    info = ORPHANET_PRODUCTS[product_key]
    output_path = DATA_DIR / info["filename"]

    if output_path.exists():
        print(f"✔ {info['filename']} già presente")
        return output_path

    print(f"⬇ Downloading {info['description']}")
    print(f"   {info['url']}")

    req = urllib.request.Request(
        info["url"],
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(req) as response:
        with open(output_path, "wb") as f:
            f.write(response.read())

    print(f"✔ Salvato in {output_path}")
    return output_path

# ----------------------------
# PARSER: product6 (disease → gene)
# ----------------------------
def parse_disease_gene(xml_path):
    xml_path = str(xml_path)  # FIX WindowsPath

    parser = etree.XMLParser(
        recover=True,
        huge_tree=True,
        no_network=True
    )
    tree = etree.parse(xml_path, parser)
    root = tree.getroot()

    rows = []

    for disorder in root.xpath(".//Disorder"):
        orpha = disorder.findtext("OrphaCode")
        disease_name = disorder.findtext("Name")

        for gene in disorder.xpath(".//Gene"):
            symbol = gene.findtext("Symbol")
            if symbol:
                rows.append({
                    "ORPHAcode": orpha,
                    "DiseaseName": disease_name,
                    "Gene": symbol
                })

    return pd.DataFrame(rows)

# ----------------------------
# MAIN PIPELINE
# ----------------------------
if __name__ == "__main__":

    # 1️⃣ Download datasets
    download_orphanet_product("product1")
    download_orphanet_product("product6")
    download_orphanet_product("product4")

    print("\n📂 Tutti i file scaricati correttamente\n")

    # 2️⃣ Parse disease → gene
    df = parse_disease_gene(DATA_DIR / "en_product6.xml")

    # 3️⃣ Save CSV
    output_csv = DATA_DIR / "orphanet_disease_gene.csv"
    df.to_csv(output_csv, index=False)
    print(f"📄 CSV salvato in: {output_csv}\n")

    # 4️⃣ Print first 3 rows
    print("🔎 Prime 3 righe del dataset:\n")
    print(df.head(3))

    # 5️⃣ Quick stats
    print("\n📊 Statistiche rapide:")
    print("Malattie uniche:", df["ORPHAcode"].nunique())
    print("Geni unici:", df["Gene"].nunique())
    print("Righe totali:", len(df))


In [ ]:

# ============================================================
# Disease–Gene–Phenotype Associations
# ============================================================

from pathlib import Path
from lxml import etree
import pandas as pd

# ============================================================
# FILE PATHS
# ============================================================

BASE = Path("data")

orphanet_csv = BASE / "orphanet_disease_gene.csv"
product4_xml = BASE / "en_product4.xml"
output_csv = BASE / "orphanet_disease_gene_hpo.csv"

# ============================================================
# LOAD DISEASE–GENE ASSOCIATIONS
# ============================================================

df = pd.read_csv(orphanet_csv)

df["ORPHAcode"] = df["ORPHAcode"].astype(str)

print("📊 Initial dataset")
print("Rows:", len(df))
print("Diseases:", df["ORPHAcode"].nunique())
print("Genes:", df["Gene"].nunique())
print("-" * 60)

# ============================================================
# EXTRACT HPO ANNOTATIONS FROM ORPHANET PRODUCT4
# ============================================================

def extract_hpo(product4_xml):
    tree = etree.parse(str(product4_xml))
    root = tree.getroot()

    rows = []

    for disorder in root.xpath(".//Disorder"):
        orpha = disorder.findtext("OrphaCode")
        if orpha is None:
            continue

        for assoc in disorder.xpath(".//HPODisorderAssociation"):
            hpo_id = assoc.findtext(".//HPOId")
            hpo_label = assoc.findtext(".//HPOTerm")
            freq = assoc.findtext(".//Frequency")

            if hpo_id:
                rows.append({
                    "ORPHAcode": str(orpha),
                    "HPO_ID": hpo_id,
                    "HPO_Label": hpo_label,
                    "Frequency": freq
                })

    return pd.DataFrame(rows)


hpo_df = extract_hpo(product4_xml)

print("🧬 Extracted HPO associations:", len(hpo_df))
print("Diseases with HPO annotations:", hpo_df["ORPHAcode"].nunique())
print("Unique HPO terms:", hpo_df["HPO_ID"].nunique())
print("-" * 60)

# ============================================================
# MERGE DISEASE–GENE AND HPO DATA USING ORPHA CODE
# ============================================================

df_hpo = df.merge(
    hpo_df,
    on="ORPHAcode",
    how="left"
)

# ============================================================
# SAVE THE MERGED DATASET
# ============================================================

df_hpo.to_csv(output_csv, index=False)

print("✅ Dataset with HPO annotations successfully created:")
print(output_csv)

print("\n🔎 First 3 rows:\n")
print(df_hpo.head(3))

# ============================================================
# FINAL DATASET STATISTICS
# ============================================================

print("\n📈 Final dataset statistics")
print("Total rows:", len(df_hpo))
print("Diseases:", df_hpo["ORPHAcode"].nunique())
print("Genes:", df_hpo["Gene"].nunique())
print("Unique HPO terms:", df_hpo["HPO_ID"].nunique())
print("Diseases with at least one HPO annotation:",
      df_hpo[df_hpo["HPO_ID"].notna()]["ORPHAcode"].nunique())


In [ ]:

# ============================================================
# Filter metabolic enzymes associated with human diseases
# from UniProtKB
# ============================================================

import pandas as pd
import re
from pathlib import Path

# ==============================
# FILE PATHS
# ==============================
DATA_DIR = Path("data")

input_file = DATA_DIR / "uniprotkb_taxonomy_id_9606_AND_reviewed_2025_12_27.tsv"
output_file = DATA_DIR / "uniprotkb_metabolic_enzymes_disease.tsv"

# ==============================
# LOAD UNIPROT TSV DATASET
# ==============================
df = pd.read_csv(input_file, sep="\t")

print(f"Total initial rows: {len(df)}")

# ==============================
# 1️⃣ CLASSICAL METABOLIC ENZYMES
#    Complete EC number X.X.X.X
# ==============================
ec_pattern = r"^\d+\.\d+\.\d+\.\d+"

df_enzymes = df[
    df["EC number"].notna() &
    df["EC number"].str.contains(ec_pattern, regex=True)
]

print(f"After complete EC number filtering: {len(df_enzymes)}")

# ==============================
# 2️⃣ PRESENCE OF CATALYTIC ACTIVITY
# ==============================
df_enzymes = df_enzymes[
    df_enzymes["Catalytic activity"].notna() &
    (df_enzymes["Catalytic activity"] != "")
]

print(f"After catalytic activity filtering: {len(df_enzymes)}")

# ==============================
# 3️⃣ INVOLVEMENT IN HUMAN DISEASES
# ==============================
df_disease = df_enzymes[
    df_enzymes["Involvement in disease"].notna() &
    (df_enzymes["Involvement in disease"] != "")
]

print(f"After disease involvement filtering: {len(df_disease)}")

# ==============================
# SAVE THE FILTERED DATASET
# ==============================
df_disease.to_csv(output_file, sep="\t", index=False)

print("=================================")
print("Extraction completed successfully ✅")
print(f"File saved to:\n{output_file}")


In [ ]:

# ============================================================
# Convert UniProtKB dataset from TSV to CSV
# ============================================================

import pandas as pd
from pathlib import Path

# --------------------------------------------------
# FILE PATHS
# --------------------------------------------------
INPUT_TSV = Path("data") / "uniprotkb_metabolic_enzymes_disease.tsv"
OUTPUT_CSV = Path("data") / "uniprotkb_metabolic_enzymes_disease.csv"

# --------------------------------------------------
# LOAD TSV FILE
# --------------------------------------------------
df = pd.read_csv(INPUT_TSV, sep="\t")

# --------------------------------------------------
# SAVE AS CSV
# --------------------------------------------------
df.to_csv(OUTPUT_CSV, index=False)

# --------------------------------------------------
# CONFIRM FILE CREATION
# --------------------------------------------------
print("✅ File saved successfully")
print("File path:", OUTPUT_CSV)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\n🔎 First 3 rows:\n")
print(df.head(3))


In [ ]:

# ============================================================
# Filter columns in the UniProtKB metabolic enzymes dataset
# ============================================================

import pandas as pd
from pathlib import Path

# ============================================================
# FILE PATHS
# ============================================================

INPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease.csv"
)

OUTPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_filtered.csv"
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("📊 Initial dataset")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("-" * 50)

# ============================================================
# DROP UNNECESSARY COLUMNS
# ============================================================

cols_to_drop = [
    "Reviewed",
    "Organism",
    "Protein names",
    "Interacts with",
    "Tissue specificity",
    "Subunit structure",
    "Disruption phenotype"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("🧹 Removed columns:", cols_to_drop)
print("Remaining columns:", len(df.columns))
print("-" * 50)

# ============================================================
# FILTER ROWS WITHOUT RHEA ID
# ============================================================

df["Rhea ID"] = df["Rhea ID"].astype(str)

df_filtered = df[
    df["Rhea ID"].notna() &
    (df["Rhea ID"] != "") &
    (df["Rhea ID"].str.lower() != "nan")
]

print("🔬 After Rhea ID filtering")
print("Remaining rows:", len(df_filtered))
print("-" * 50)

# ============================================================
# SAVE FILTERED DATASET
# ============================================================

df_filtered.to_csv(OUTPUT_FILE, index=False)

print("✅ Filtered file saved to:")
print(OUTPUT_FILE)

# ============================================================
# FINAL CHECK
# ============================================================

print("\n🔎 First 3 rows:")
print(df_filtered.head(3))

print("\n📈 Final statistics:")
print("Rows:", len(df_filtered))
print("Unique EC numbers:", df_filtered["EC number"].nunique())
print("Unique Rhea IDs:", df_filtered["Rhea ID"].nunique())
print("Unique genes:", df_filtered["Gene Names"].nunique())


In [ ]:

# ============================================================
# COFACTOR FILTERING AND NORMALIZATION
# ============================================================

import pandas as pd
import re
from pathlib import Path

# ============================================================
# FILE PATHS
# ============================================================

INPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_filtered.csv"
)

OUTPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_filtered_clean_cofactor.csv"
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("📊 Dataset loaded:", len(df), "rows")

# ============================================================
# COFACTOR VOCABULARY
# ============================================================

VALID_COFACTORS = {
    "FAD", "FMN", "NAD(+)", "NADP(+)",
    "pyridoxal 5'-phosphate",
    "thiamine diphosphate",
    "biotin",
    "(R)-lipoate",
    "heme", "heme b", "heme c"
}

METAL_PATTERN = re.compile(
    r"\b(Mg|Mn|Fe|Zn|Ca|Cu|Co|Ni|Cd|K)\s*\(\d+\+\)|"
    r"\bFe cation\b|"
    r"\ba divalent metal cation\b",
    re.IGNORECASE
)

# ============================================================
# EXTRACT AND NORMALIZE COFACTORS
# ============================================================

def extract_and_normalize_cofactor(text):
    if pd.isna(text):
        return None

    # 1️⃣ Extract all Name=... entries
    names = re.findall(r"Name=([^;]+)", text)

    found = set()

    for n in names:
        n = n.strip()

        # Organic cofactors
        if n in VALID_COFACTORS:
            found.add(n)

        # Metal cofactors
        elif METAL_PATTERN.search(n):
            found.add("METAL")

    if not found:
        return None

    return ";".join(sorted(found))

# ============================================================
# APPLY COFACTOR NORMALIZATION
# ============================================================

df["Cofactor_normalized"] = df["Cofactor"].apply(extract_and_normalize_cofactor)

# ============================================================
# SAVE DATASET
# ============================================================

df.to_csv(OUTPUT_FILE, index=False)

print("✅ File saved successfully:")
print(OUTPUT_FILE)

# ============================================================
# CHECK RESULTS
# ============================================================

print("\n🔎 Examples:")
print(df[["Cofactor", "Cofactor_normalized"]].dropna().head(10))


In [ ]:

# ============================================================
# PATHWAY FILTERING AND CLEANING
# ============================================================

import pandas as pd
import re
from pathlib import Path

# ============================================================
# FILE PATHS
# ============================================================

INPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_filtered_clean_cofactor.csv"
)

OUTPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_filtered_clean_cofactor_pathway.csv"
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("📊 Dataset loaded")
print("Rows:", len(df))
print("-" * 50)

# ============================================================
# CLEAN PATHWAY COLUMN (PRECISION MODE)
# ============================================================

def clean_pathway(text):
    if pd.isna(text):
        return text

    # 1️⃣ Remove all {ECO:...} blocks
    text = re.sub(r"\{ECO:[^}]*\}", "", text)

    # 2️⃣ Remove PATHWAY: prefix
    text = re.sub(r"\bPATHWAY:\s*", "", text)

    # 3️⃣ Normalize semicolon separators
    text = re.sub(r"\s*;\s*", ";", text)

    # 4️⃣ Final cleanup
    text = text.strip(" ;")

    return text

# ============================================================
# APPLY PATHWAY CLEANING
# ============================================================

df["Pathway"] = df["Pathway"].apply(clean_pathway)

# ============================================================
# SAVE DATASET
# ============================================================

df.to_csv(OUTPUT_FILE, index=False)

print("✅ File saved with cleaned Pathway column:")
print(OUTPUT_FILE)

# ============================================================
# CHECK RESULTS
# ============================================================

print("\n🔎 Examples of cleaned Pathway entries:")
print(df["Pathway"].dropna().head(5))


In [ ]:

# ============================================================
# FINAL UNIPROT DATASET CLEANING
# ============================================================

import pandas as pd
from pathlib import Path
import re

# ============================================================
# FILE PATHS
# ============================================================

INPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_filtered_clean_cofactor_pathway.csv"
)

OUTPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_FINAL.csv"
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("📊 Dataset loaded")
print("Rows:", len(df))
print("Initial columns:", len(df.columns))
print("-" * 60)

# ============================================================
# DROP COLUMNS
# ============================================================

cols_to_drop = [
    "Activity regulation",
    "Function [CC]",
    "Polymorphism",
    "Length",
    "Mass",
    "Subcellular location [CC]",
    "Subunit structure"
]

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("🧹 Removed columns:")
for c in cols_to_drop:
    if c in df.columns:
        print("  ❌", c)

# ============================================================
# CLEAN INVOLVEMENT IN DISEASE COLUMN
# ============================================================

if "Involvement in disease" in df.columns:
    df["Involvement in disease"] = (
        df["Involvement in disease"]
        .astype(str)
        .str.replace(r"^DISEASE:\s*", "", regex=True)
        .str.strip()
    )

print("🧬 'Involvement in disease' column cleaned")

# ============================================================
# SAVE FINAL DATASET
# ============================================================

df.to_csv(OUTPUT_FILE, index=False)

print("-" * 60)
print("✅ Final file saved successfully:")
print(OUTPUT_FILE)

# ============================================================
# FINAL STATISTICS
# ============================================================

print("\n📈 Final statistics")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique genes:", df["Gene Names"].nunique() if "Gene Names" in df.columns else "NA")
print("Unique EC numbers:", df["EC number"].nunique() if "EC number" in df.columns else "NA")
print("Unique Rhea IDs:", df["Rhea ID"].nunique() if "Rhea ID" in df.columns else "NA")


In [ ]:

# ============================================================
# REORDER COLUMNS IN THE FINAL UNIPROT DATASET
# ============================================================

import pandas as pd
from pathlib import Path

# ============================================================
# FILE PATHS
# ============================================================

INPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_FINAL.csv"
)

OUTPUT_FILE = Path(
    "data/uniprotkb_metabolic_enzymes_disease_FINAL_reordered.csv"
)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("📊 Dataset loaded")
print("Original columns:")
print(df.columns.tolist())
print("-" * 60)

# ============================================================
# MOVE 'Gene Names' AFTER 'Entry'
# ============================================================

cols = df.columns.tolist()

if "Entry" not in cols or "Gene Names" not in cols:
    raise ValueError("❌ Columns 'Entry' or 'Gene Names' not found")

# Remove Gene Names
cols.remove("Gene Names")

# Insert Gene Names immediately after Entry
entry_index = cols.index("Entry")
cols.insert(entry_index + 1, "Gene Names")

# Reorder dataframe
df = df[cols]

print("✅ Columns reordered successfully")
print("New column order:")
print(df.columns.tolist())

# ============================================================
# SAVE REORDERED DATASET
# ============================================================

df.to_csv(OUTPUT_FILE, index=False)

print("-" * 60)
print("💾 File saved to:")
print(OUTPUT_FILE)


In [ ]:

# ============================================================
# EXTRACT EC NUMBER, COFACTOR, RHEA ID, AND PATHWAY COLUMNS
# FROM THE FINAL REORDERED UNIPROT DATASET
# ============================================================

import pandas as pd

# ============================================================
# FILE PATHS
# ============================================================

input_path = 'data/uniprotkb_metabolic_enzymes_disease_FINAL_reordered.csv'
output_path = 'data/cofactor_rhea_EC_path.csv'

try:
    # Load the file (sep=None automatically detects , or ;)
    df = pd.read_csv(input_path, sep=None, engine='python')

    # Updated list of columns to extract
    columns_to_extract = ['EC number', 'Cofactor', 'Rhea ID', 'Pathway']

    # Check whether all required columns exist in the file
    existing_columns = [col for col in columns_to_extract if col in df.columns]

    if len(existing_columns) < len(columns_to_extract):
        missing = set(columns_to_extract) - set(existing_columns)
        print(f"Warning: The following columns were not found: {missing}")
        print(f"Columns actually present in the file: {list(df.columns)}")
    else:
        # Extract the 4 columns
        df_final = df[columns_to_extract]

        # Save the result
        df_final.to_csv(output_path, index=False)

        print("Extraction completed successfully!")
        print(f"File saved to: {output_path}")
        print("\nFirst 5 rows of the new file:")
        print(df_final.head())

except Exception as e:
    print(f"Error during processing: {e}")


In [ ]:

# ============================================================
# EXTRACT ChEBI IDs AND COFACTOR NAMES
# ============================================================

import pandas as pd

# ============================================================
# FILE PATHS
# ============================================================

input_path = 'data/uniprotkb_metabolic_enzymes_disease_FINAL_reordered.csv'
output_path = 'data/cofactor_rhea_EC_cleaned.csv'

# ============================================================
# EXTRACT COFACTOR INFORMATION
# ============================================================

def extract_cofactor_info(cell):
    """
    Function to split the cofactor string into ChEBI IDs and names.
    Removes notes and preserves the original order.
    """
    if pd.isna(cell) or cell == "":
        return "", ""

    # Split by ';;', which separates different cofactors or notes
    parts = str(cell).split(';;')

    names = []
    chebi_ids = []

    for part in parts:
        part = part.strip()

        # Ignore empty parts or those starting with "Note="
        if not part or part.startswith("Note="):
            continue

        # Each valid part follows the format "Name;CHEBI:ID"
        # Split by ';' to separate the name and ID
        sub_parts = part.split(';')
        if len(sub_parts) >= 2:
            name = sub_parts[0].strip()
            chebi = sub_parts[1].strip()

            # Check whether the second part is a ChEBI ID
            if "CHEBI:" in chebi:
                names.append(name)
                chebi_ids.append(chebi)

    # Join the results using semicolons
    return ";".join(chebi_ids), ";".join(names)


try:
    # 1. Load the dataset
    df = pd.read_csv(input_path, sep=None, engine='python')

    # 2. Apply the extraction function
    # Create two new columns by applying the function to 'Cofactor'
    df[['ChEBI cofactor', 'Cofactor name']] = df['Cofactor'].apply(
        lambda x: pd.Series(extract_cofactor_info(x))
    )

    # 3. Select and reorder the final columns
    columns_to_keep = ['EC number', 'Rhea ID', 'Pathway', 'ChEBI cofactor', 'Cofactor name']

    # Check whether the original columns exist
    existing_cols = [c for c in columns_to_keep if c in df.columns or c in ['ChEBI cofactor', 'Cofactor name']]
    df_final = df[existing_cols]

    # 4. Save the dataset
    df_final.to_csv(output_path, index=False)

    print("Process completed successfully!")
    print(f"File saved to: {output_path}")
    print("\nExample of the results:")
    print(df_final[['ChEBI cofactor', 'Cofactor name']].head(10))

except Exception as e:
    print(f"Error: {e}")


In [ ]:

# =====================================================
# COFACTOR CLEANING AND BIOLOGICAL VALIDATION
# Date: 27/01/2026
# =====================================================

import pandas as pd

# =====================================================
# 1. LOAD FILE
# =====================================================
input_path = "data/cofactor_rhea_EC_cleaned.csv"

try:
    df = pd.read_csv(input_path)
except Exception:
    df = pd.read_csv(input_path, encoding="latin1")

# =====================================================
# 2. NORMALIZE TEXT COLUMNS
# =====================================================
df["Cofactor name"] = (
    df["Cofactor name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["ChEBI cofactor"] = (
    df["ChEBI cofactor"]
    .astype(str)
    .str.strip()
)

# =====================================================
# 3. SPLIT MULTIPLE COFACTORS (CHEBI-BASED)
# =====================================================
df["Cofactor_list"] = df["Cofactor name"].str.split(";")
df["ChEBI_list"] = df["ChEBI cofactor"].str.split(";")

df = df.explode(["Cofactor_list", "ChEBI_list"])

df["Cofactor_list"] = df["Cofactor_list"].str.strip()
df["ChEBI_list"] = df["ChEBI_list"].str.strip()

# =====================================================
# 4. DEFINE VALID CHEBI SETS (BIOLOGICALLY CURATED)
# =====================================================

# --- Organic cofactors ---
ORGANIC_CHEBI = {
    "CHEBI:57692",   # FAD
    "CHEBI:58210",   # FMN
    "CHEBI:57540",   # NAD(+)
    "CHEBI:58349",   # NADP(+)
    "CHEBI:597326",  # pyridoxal 5'-phosphate (PLP)
    "CHEBI:15346",   # thiamine diphosphate
    "CHEBI:15956",   # biotin
    "CHEBI:29985",   # (R)-lipoate
    "CHEBI:16304",   # adenosylcobalamin
    "CHEBI:16309",   # methylcobalamin
    "CHEBI:30413",   # heme
    "CHEBI:60344",   # heme b
    "CHEBI:61717",   # heme c
    "CHEBI:16856",   # glutathione
    "CHEBI:83088"    # dipyrromethane
}

# --- Metals and metal clusters ---
METAL_CHEBI = {
    "CHEBI:18420",   # Mg(2+)
    "CHEBI:29035",   # Mn(2+)
    "CHEBI:29105",   # Zn(2+)
    "CHEBI:29108",   # Ca(2+)
    "CHEBI:29033",   # Fe(2+)
    "CHEBI:29034",   # Fe(3+)
    "CHEBI:24875",   # Fe cation
    "CHEBI:29036",   # Cu(2+)
    "CHEBI:23378",   # Cu cation
    "CHEBI:48828",   # Co(2+)
    "CHEBI:49786",   # Ni(2+)
    "CHEBI:49415",   # Cd(2+)
    "CHEBI:29103",   # K(+)
    "CHEBI:49883",   # [4Fe-4S] cluster
    "CHEBI:47854",   # [2Fe-2S] cluster
    "CHEBI:60240",   # divalent metal cation (unspecified)
    "CHEBI:190135"   # Mo-molybdopterin related
}

VALID_CHEBI = ORGANIC_CHEBI.union(METAL_CHEBI)

# =====================================================
# 5. REMOVE NON-COFACTORS
# =====================================================
EXCLUDE_CHEBI = {
    "CHEBI:15422",  # ATP
    "CHEBI:15361",  # pyruvate
    "CHEBI:28115"   # inositol hexakisphosphate
}

df = df[
    df["ChEBI_list"].isin(VALID_CHEBI)
    & ~df["ChEBI_list"].isin(EXCLUDE_CHEBI)
]

# =====================================================
# 6. ASSIGN COFACTOR TYPE
# =====================================================
df["Cofactor_type"] = df["ChEBI_list"].apply(
    lambda x: "metal" if x in METAL_CHEBI else "organic_cofactor"
)

# =====================================================
# 7. FINAL TIDY TABLE
# =====================================================
final_df = df[
    [
        "EC number",
        "Rhea ID",
        "Pathway",
        "Cofactor_list",
        "Cofactor_type",
        "ChEBI_list"
    ]
].rename(
    columns={
        "Cofactor_list": "Cofactor",
        "ChEBI_list": "CHEBI"
    }
)

# =====================================================
# 8. SAVE OUTPUT
# =====================================================
output_path = "data/cofactor_FINAL_CORRECT.csv"
final_df.to_csv(output_path, index=False)

print("✅ Final cleaned dataset created successfully")
print("📁 File saved to:", output_path)


In [ ]:

# =====================================================
# MERGE THE MAIN DATASET WITH COFACTOR INFORMATION
# =====================================================

import pandas as pd

# =====================================================
# 1. LOAD DATASETS
# =====================================================
main_path = "data/dataset_finale_COMPLETATO.csv"
cofactor_path = "data/cofactor_FINAL_CORRECT.csv"

df_main = pd.read_csv(main_path)
df_cof = pd.read_csv(cofactor_path)

# =====================================================
# 2. NORMALIZE JOIN KEYS
# =====================================================
df_main["Rhea_ID"] = df_main["Rhea_ID"].astype(str).str.strip()
df_main["EC"] = df_main["EC"].astype(str).str.strip()

df_cof["Rhea ID"] = df_cof["Rhea ID"].astype(str).str.strip()
df_cof["EC number"] = df_cof["EC number"].astype(str).str.strip()

# =====================================================
# 3. MERGE (LEFT JOIN, MANY-TO-MANY SAFE)
# =====================================================
merged_df = df_main.merge(
    df_cof,
    how="left",
    left_on=["Rhea_ID", "EC"],
    right_on=["Rhea ID", "EC number"]
)

# =====================================================
# 4. DROP UNUSED / DUPLICATE COLUMNS
# =====================================================
merged_df = merged_df.drop(
    columns=[
        "Rhea ID",
        "EC number",
        "PhysiologicalDirection"  # Removed as requested
    ],
    errors="ignore"
)

# =====================================================
# 5. RENAME COFACTOR CHEBI COLUMN
# =====================================================
merged_df = merged_df.rename(
    columns={"CHEBI": "CHEBI_Cofactor"}
)

# =====================================================
# 6. SELECT & ORDER FINAL COLUMNS
# =====================================================
final_columns = [
    "ORPHAcode",
    "DiseaseName",
    "Gene",
    "Entry",
    "Rhea_ID",
    "Reaction",
    "EC",
    "ChEBI substruct",
    "ChEBI product",
    "Cofactor",
    "Cofactor_type",
    "CHEBI_Cofactor",
    "HPO_ID",
    "HPO_Label"
]

# Keep only columns that are actually present
final_columns = [c for c in final_columns if c in merged_df.columns]
final_df = merged_df[final_columns]

# =====================================================
# 7. SAVE FINAL DATASET
# =====================================================
output_path = "data/dataset_finale_CON_COFATTORI.csv"
final_df.to_csv(output_path, index=False)

print("✅ Final dataset created successfully")
print("📁 File saved to:", output_path)
print("🔬 Number of final rows:", final_df.shape[0])


In [ ]:

# =====================================================
# STATISTICAL ANALYSIS OF THE DATASETS
# =====================================================

import pandas as pd

# =====================================================
# FILE PATHS
# =====================================================
main_path = "data/dataset_finale_COMPLETATO.csv"
cofactor_path = "data/cofactor_FINAL_CORRECT.csv"

# =====================================================
# LOAD DATASETS
# =====================================================
df_main = pd.read_csv(main_path)
df_cof = pd.read_csv(cofactor_path)

# =====================================================
# NORMALIZE COLUMNS
# =====================================================
df_main["Rhea_ID"] = df_main["Rhea_ID"].astype(str).str.strip()
df_main["EC"] = df_main["EC"].astype(str).str.strip()

df_cof["Rhea ID"] = df_cof["Rhea ID"].astype(str).str.strip()
df_cof["EC number"] = df_cof["EC number"].astype(str).str.strip()

# =====================================================
# MAIN DATASET ANALYSIS
# =====================================================
main_stats = {
    "dataset": "dataset_finale_COMPLETATO",
    "rows": df_main.shape[0],
    "unique_Rhea_ID": df_main["Rhea_ID"].nunique(),
    "unique_EC": df_main["EC"].nunique(),
    "unique_Rhea_EC": df_main[["Rhea_ID", "EC"]].drop_duplicates().shape[0]
}

# =====================================================
# COFACTOR DATASET ANALYSIS
# =====================================================
cofactor_stats = {
    "dataset": "cofactor_FINAL_CORRECT",
    "rows": df_cof.shape[0],
    "unique_Rhea_ID": df_cof["Rhea ID"].nunique(),
    "unique_EC": df_cof["EC number"].nunique(),
    "unique_Rhea_EC": df_cof[["Rhea ID", "EC number"]].drop_duplicates().shape[0]
}

# =====================================================
# PRINT RESULTS
# =====================================================
stats_df = pd.DataFrame([main_stats, cofactor_stats])

print("📊 DATASET SUMMARY STATISTICS")
print("-" * 50)
print(stats_df)


In [ ]:

# =====================================================
# STATISTICAL ANALYSIS OF COFACTOR PRESENCE IN REACTIONS
# =====================================================

import pandas as pd

# =====================================================
# 1. LOAD DATASET
# =====================================================
path = "data/dataset_finale_CON_COFATTORI.csv"
df = pd.read_csv(path)

# =====================================================
# 2. DEFINE THE REACTION UNIT
# =====================================================
# Use Rhea_ID and EC as the reaction identifier
# (EC can be included for greater granularity)

reaction_key = ["Rhea_ID", "EC"]


# =====================================================
# 3. IDENTIFY REACTIONS WITH AND WITHOUT COFACTORS
# =====================================================
# A reaction has a cofactor if at least one row
# contains a non-null value in the Cofactor column

cofactor_presence = (
    df
    .groupby(reaction_key)["Cofactor"]
    .apply(lambda x: x.notna().any())
    .reset_index(name="has_cofactor")
)

# =====================================================
# 4. COUNT REACTIONS
# =====================================================
total_reactions = cofactor_presence.shape[0]

reactions_with_cofactor = cofactor_presence["has_cofactor"].sum()
reactions_without_cofactor = total_reactions - reactions_with_cofactor

# =====================================================
# 5. CALCULATE PERCENTAGES
# =====================================================
perc_with = reactions_with_cofactor / total_reactions * 100
perc_without = reactions_without_cofactor / total_reactions * 100

# =====================================================
# 6. PRINT RESULTS
# =====================================================
print("📊 COFACTOR ANALYSIS BY RHEA_ID")
print("-" * 40)
print(f"Total reactions (Rhea_ID): {total_reactions:,}")
print(f"Reactions WITH cofactors:    {reactions_with_cofactor:,} ({perc_with:.2f}%)")
print(f"Reactions WITHOUT cofactors: {reactions_without_cofactor:,} ({perc_without:.2f}%)")

df.groupby(["Rhea_ID", "EC"])["Cofactor"].nunique().value_counts().sort_index()
cofactor_presence[~cofactor_presence["has_cofactor"]].head()

print(df.groupby(["Rhea_ID", "EC"])["Cofactor"].nunique().value_counts().sort_index())
print(cofactor_presence[~cofactor_presence["has_cofactor"]].head())


In [ ]:

# =====================================================
# EXTRACT REACTIONS WITHOUT COFACTOR ANNOTATIONS
# =====================================================

import pandas as pd

# =====================================================
# FILE PATHS
# =====================================================
input_path = "data/dataset_finale_CON_COFATTORI.csv"
output_path = "data/reazioni_senza_cofattore.csv"

# =====================================================
# 1. LOAD DATASET
# =====================================================
df = pd.read_csv(input_path)

# =====================================================
# 2. NORMALIZE JOIN KEYS
# =====================================================
df["Rhea_ID"] = df["Rhea_ID"].astype(str).str.strip()
df["EC"] = df["EC"].astype(str).str.strip()

# =====================================================
# 3. IDENTIFY (Rhea_ID, EC) PAIRS WITHOUT COFACTORS
# =====================================================
# A reaction has no cofactor annotation if all rows
# have missing (NaN) or empty Cofactor values

no_cofactor_pairs = (
    df
    .groupby(["Rhea_ID", "EC"])["Cofactor"]
    .apply(lambda x: x.isna().all() or (x.astype(str).str.strip() == "").all())
    .reset_index(name="no_cofactor")
)

# Keep only pairs without cofactors
no_cofactor_pairs = no_cofactor_pairs[no_cofactor_pairs["no_cofactor"]]

# =====================================================
# 4. FILTER THE ORIGINAL DATASET
# =====================================================
df_no_cofactor = df.merge(
    no_cofactor_pairs[["Rhea_ID", "EC"]],
    on=["Rhea_ID", "EC"],
    how="inner"
)

# =====================================================
# 5. SELECT REQUIRED COLUMNS
# =====================================================
df_no_cofactor = df_no_cofactor[
    ["Entry", "Rhea_ID", "EC", "Cofactor", "Cofactor_type", "CHEBI_Cofactor"]
].drop_duplicates()

# =====================================================
# 6. SAVE DATASET
# =====================================================
df_no_cofactor.to_csv(output_path, index=False)

# =====================================================
# 7. PRINT EXTRACTION REPORT
# =====================================================
print("📊 EXTRACTION COMPLETED")
print("-" * 40)
print(f"Reactions without cofactors (Rhea_ID–EC): {no_cofactor_pairs.shape[0]}")
print(f"Rows in the extracted file:               {df_no_cofactor.shape[0]}")
print(f"File saved to: {output_path}")


In [ ]:

# =====================================================
# RETRIEVE COFACTORS FROM UNIPROT ANNOTATIONS
# =====================================================

import pandas as pd
import requests
import time

# =====================================================
# FILE PATHS
# =====================================================
path_input = 'data/reazioni_senza_cofattore.csv'
path_output = 'data/cofattori_recuperati.csv'

# =====================================================
# UNIPROT COFACTOR RETRIEVAL FUNCTION
# =====================================================
def fetch_uniprot_cofactor(entry):
    """
    Retrieve protein cofactors annotated in UniProt
    (commentType = COFACTOR).
    Return (cofactor_names, chebi_ids) or (None, None).
    """
    url = f"https://rest.uniprot.org/uniprotkb/{entry}.json"
    names = []
    chebis = []

    try:
        r = requests.get(url, timeout=10)
        if r.status_code != 200:
            return None, None

        data = r.json()

        for comment in data.get("comments", []):
            if comment.get("commentType") == "COFACTOR":
                for cof in comment.get("cofactors", []):
                    name = cof.get("name")
                    chebi = cof.get("cofactorCrossReference", {}).get("id")
                    if name:
                        names.append(name)
                    if chebi:
                        chebis.append(chebi)

        return (
            "; ".join(sorted(set(names))) if names else None,
            "; ".join(sorted(set(chebis))) if chebis else None
        )

    except Exception as e:
        print(f"UniProt error for {entry}: {e}")
        return None, None

# =====================================================
# LOAD DATA (COMMA-SEPARATED CSV)
# =====================================================
df_mancanti = pd.read_csv(path_input)  # Default separator: ','

# Clean column names
df_mancanti.columns = df_mancanti.columns.str.strip()

# Check whether the Entry column exists
if "Entry" not in df_mancanti.columns:
    raise ValueError(
        f"'Entry' column not found. Available columns: {df_mancanti.columns.tolist()}"
    )

print(f"Starting UniProt retrieval for {len(df_mancanti)} entries...")

# =====================================================
# OUTPUT COLUMNS (UNIPROT-SPECIFIC)
# =====================================================
df_mancanti["UniProt_Cofactor"] = None
df_mancanti["UniProt_ChEBI"] = None

# =====================================================
# BATCH QUERY PROCESSING
# =====================================================
for i, row in df_mancanti.iterrows():
    entry = str(row["Entry"]).strip()

    if not entry or entry.lower() == "nan":
        continue

    name, chebi = fetch_uniprot_cofactor(entry)

    df_mancanti.at[i, "UniProt_Cofactor"] = name
    df_mancanti.at[i, "UniProt_ChEBI"] = chebi

    if i % 50 == 0:
        print(f"Processed {i}/{len(df_mancanti)} entries")

    time.sleep(0.2)  # Respect UniProt rate limits

# =====================================================
# SAVE OUTPUT (SEMICOLON-SEPARATED CSV)
# =====================================================
df_mancanti.to_csv(path_output, index=False, sep=';')

# =====================================================
# FINAL REPORT
# =====================================================
print("\n✅ COMPLETED")
print(f"File saved to: {path_output}")
print(f"Entries with UniProt cofactor annotations: {df_mancanti['UniProt_Cofactor'].notna().sum()}")


In [ ]:

# =====================================================
# MERGE RECOVERED UNIPROT COFACTORS WITH THE MAIN DATASET
# =====================================================

import pandas as pd
import numpy as np

# =====================================================
# FILE PATHS
# =====================================================
file_recupero_path = "data/cofattori_recuperati.csv"
file_dataset_path  = "data/dataset_finale_CON_COFATTORI.csv"
file_output_path   = "data/dataset_finale_AGGIORNATO.csv"

# =====================================================
# LOAD FILES (CORRECT SEPARATORS)
# =====================================================
df_rec = pd.read_csv(
    file_recupero_path,
    sep=";",
    engine="python",
    on_bad_lines="skip"
)

df_main = pd.read_csv(
    file_dataset_path,
    sep=",",          # Correct separator for the main dataset
    engine="python",
    on_bad_lines="skip"
)

# =====================================================
# CLEAN COLUMN NAMES
# =====================================================
df_rec.columns  = df_rec.columns.str.strip()
df_main.columns = df_main.columns.str.strip()

print("RECOVERED COFACTOR COLUMNS:")
print(df_rec.columns.tolist())
print("\nMAIN DATASET COLUMNS:")
print(df_main.columns.tolist())

# =====================================================
# CHECK REQUIRED COLUMNS
# =====================================================
required_rec = {
    "Entry", "Rhea_ID", "EC",
    "UniProt_Cofactor", "UniProt_ChEBI"
}
required_main = {
    "Entry", "Rhea_ID", "EC",
    "Cofactor", "CHEBI_Cofactor"
}

missing_rec  = required_rec - set(df_rec.columns)
missing_main = required_main - set(df_main.columns)

if missing_rec:
    raise ValueError(f"Missing columns in cofattori_recuperati.csv: {missing_rec}")
if missing_main:
    raise ValueError(f"Missing columns in the main dataset: {missing_main}")

# =====================================================
# NORMALIZE JOIN KEYS
# =====================================================
for col in ["Entry", "Rhea_ID", "EC"]:
    df_rec[col]  = df_rec[col].astype(str).str.strip()
    df_main[col] = df_main[col].astype(str).str.strip()

# =====================================================
# SELECT UNIPROT COLUMNS
# =====================================================
df_rec_clean = (
    df_rec[
        ["Entry", "Rhea_ID", "EC",
         "UniProt_Cofactor", "UniProt_ChEBI"]
    ]
    .drop_duplicates()
)

# =====================================================
# MERGE (REACTION-LEVEL)
# =====================================================
df_merged = df_main.merge(
    df_rec_clean,
    on=["Entry", "Rhea_ID", "EC"],
    how="left"
)

# =====================================================
# IMPUTE COFACTORS (ONLY WHEN MISSING)
# =====================================================
df_merged["Cofactor"] = (
    df_merged["Cofactor"]
    .replace(r"^\s*$", np.nan, regex=True)
    .fillna(df_merged["UniProt_Cofactor"])
)

df_merged["CHEBI_Cofactor"] = (
    df_merged["CHEBI_Cofactor"]
    .replace(r"^\s*$", np.nan, regex=True)
    .fillna(df_merged["UniProt_ChEBI"])
)

# =====================================================
# REMOVE TEMPORARY COLUMNS
# =====================================================
df_merged.drop(
    columns=["UniProt_Cofactor", "UniProt_ChEBI"],
    inplace=True
)

# =====================================================
# SAVE UPDATED DATASET
# =====================================================
df_merged.to_csv(file_output_path, sep=",", index=False)

print("\n✅ UPDATE COMPLETED")
print(f"File saved to:\n{file_output_path}")

# =====================================================
# FINAL STATISTICS
# =====================================================
tot_rhea = df_merged["Rhea_ID"].nunique()
missing_rhea = (
    df_merged
    .loc[df_merged["Cofactor"].isna(), "Rhea_ID"]
    .nunique()
)

print("\n📊 COFACTOR STATISTICS")
print(f"Total Rhea_IDs:                   {tot_rhea}")
print(f"Rhea_IDs still without cofactor: {missing_rhea}")
print(f"Cofactor coverage: {(1 - missing_rhea/tot_rhea)*100:.2f}%")


In [ ]:

# =====================================================
# PREPARE THE LARGE DATASET FOR MAUDE - STOICHIOMETRY
# =====================================================

import pandas as pd
import re

# =====================================================
# 1. FILE PATHS
# =====================================================
input_file = "data/dataset_finale_AGGIORNATO.csv"
output_file = "data/dataset_finale_Stoichiometry.csv"

# =====================================================
# 2. LOAD DATASET
# =====================================================
try:
    df = pd.read_csv(input_file, sep=';', encoding='utf-8', dtype=str)
except:
    df = pd.read_csv(input_file, sep=',', encoding='utf-8', dtype=str)

print("Processing separate 'Names' and 'Values'...")

# =====================================================
# 3. PARSING FUNCTIONS
# =====================================================

def split_reaction_data(reaction_str):
    """
    Return:
    - Substrate names
    - Substrate stoichiometry
    - Product names
    - Product stoichiometry
    - Dictionary mapping molecule → coefficient
    """
    if pd.isna(reaction_str) or str(reaction_str).strip() == '':
        return "", "", "", "", {}

    parts = str(reaction_str).split(' = ')
    if len(parts) != 2:
        return "", "", "", "", {}

    reaction_dict = {}

    def parse_side(side_str):
        molecules = side_str.split(' + ')
        names, values = [], []

        for mol in molecules:
            mol = mol.strip()
            match = re.match(r'^(\d+|\(n\)|\(n-1\)|n)\s+(.*)', mol)

            if match:
                coef = match.group(1)
                name = match.group(2)
            else:
                coef = "1"
                name = mol

            names.append(name)
            values.append(coef)
            reaction_dict[name] = coef

        return ";".join(names), ";".join(values)

    sub_names, sub_vals = parse_side(parts[0])
    prod_names, prod_vals = parse_side(parts[1])

    return sub_names, sub_vals, prod_names, prod_vals, reaction_dict


def get_cofactor_values(cofactor_str, reaction_dict):
    """
    0 = catalytic / implicit
    """
    if pd.isna(cofactor_str) or str(cofactor_str).strip() in ['', 'ND', 'nan']:
        return "ND"

    cofactors = str(cofactor_str).split(';')
    values = []

    for cof in cofactors:
        cof = cof.strip()
        if not cof:
            continue

        if cof in reaction_dict:
            values.append(reaction_dict[cof])
        else:
            found = "0"
            for rxn_mol, rxn_coef in reaction_dict.items():
                if cof in rxn_mol or rxn_mol in cof:
                    found = rxn_coef
                    break
            values.append(found)

    return ";".join(values)

# =====================================================
# 4. APPLY PARSING FUNCTIONS
# =====================================================
s_names, s_vals, p_names, p_vals, c_vals = [], [], [], [], []

for _, row in df.iterrows():
    reaction = row.get('Reaction', '')
    cofactor = row.get('Cofactor', '')

    sn, sv, pn, pv, r_dict = split_reaction_data(reaction)
    cv = get_cofactor_values(cofactor, r_dict)

    s_names.append(sn)
    s_vals.append(sv)
    p_names.append(pn)
    p_vals.append(pv)
    c_vals.append(cv)

# =====================================================
# 5. CREATE NEW COLUMNS
# =====================================================
df['Substrates_names'] = s_names
df['Reactants_Stoichiometry'] = s_vals
df['product_nams'] = p_names
df['Products_Stoichiometry'] = p_vals
df['Cofactor_Stoichiometry'] = c_vals

# =====================================================
# 6. RENAME COLUMNS
# =====================================================
df = df.rename(columns={
    'ChEBI substruct': 'ChEBI substrates',
    'CHEBI_Cofactor': 'ChEBI cofactor'
})

# =====================================================
# 7. FINAL COLUMN ORDER (MAUDE-READY)
# =====================================================
desired_order = [
    'ORPHAcode',
    'DiseaseName',
    'Gene',
    'Entry',
    'Rhea_ID',
    'Reaction',
    'EC',
    'Substrates_names',
    'ChEBI substrates',
    'Reactants_Stoichiometry',
    'product_nams',
    'ChEBI product',
    'Products_Stoichiometry',
    'Cofactor',
    'Cofactor_type',
    'ChEBI cofactor',
    'Cofactor_Stoichiometry',
    'PhysiologicalDirection',
    'HPO_ID',
    'HPO_Label'
]

df_final = df[[c for c in desired_order if c in df.columns]]

# =====================================================
# 8. SAVE DATASET
# =====================================================
df_final.to_csv(output_file, sep=';', index=False)

print("✅ FILE CREATED:", output_file)
print("Example:")
print("Substrates:", df_final['Substrates_names'].iloc[0])
print("Stoichiometry:", df_final['Reactants_Stoichiometry'].iloc[0])
print("Cofactors:", df_final['Cofactor_Stoichiometry'].iloc[0], "(0 = catalytic)")


In [ ]:

# =====================================================
# REMOVE DUPLICATE ROWS FROM THE STOICHIOMETRY DATASET
# =====================================================

import pandas as pd
import os

# =====================================================
# FILE PATHS
# =====================================================
path_input = "data/dataset_finale_Stoichiometry.csv"
path_output = "data/dataset_finale_Stoichiometry_CLEAN.csv"

print("--- STARTING DUPLICATE REMOVAL ---")

try:
    # 1. LOAD DATASET
    # Use consistent handling of 'ND' and other missing values
    # to identify logically equivalent duplicate rows
    valori_nulli = ['ND', 'nd', 'Nd', 'N/A', 'n/a', '-', '', ' ']

    df = pd.read_csv(
        path_input,
        sep=';',
        dtype=str,
        engine='python',
        na_values=valori_nulli,
        keep_default_na=True
    )

    # Strip whitespace from all string columns
    # to avoid inconsistencies when identifying duplicates
    df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

    # Initial row count
    totale_iniziale = len(df)
    print(f"📊 Total initial rows: {totale_iniziale}")

    # 2. REMOVE DUPLICATES
    # keep='first' preserves the first occurrence
    # and removes subsequent identical rows
    df_clean = df.drop_duplicates(keep='first')

    # 3. CALCULATE STATISTICS
    totale_finale = len(df_clean)
    duplicati_rimossi = totale_iniziale - totale_finale

    # 4. SAVE CLEANED DATASET
    # na_rep='ND' is optional if missing values should be saved as 'ND'.
    # Here, missing values are saved as empty cells.
    df_clean.to_csv(path_output, index=False, sep=';')

    print("-" * 30)
    print("✅ OPERATION COMPLETED!")
    print(f"🗑️ Duplicates removed: {duplicati_rimossi}")
    print(f"💾 Remaining unique rows: {totale_finale}")
    print("-" * 30)
    print(f"📁 File saved to: {path_output}")

except Exception as e:
    print(f"❌ Error during processing: {e}")


In [ ]:

# =====================================================
# STATISTICAL ANALYSIS OF COFACTOR PRESENCE IN REACTIONS
# =====================================================

import pandas as pd

# ===============================
# 1. LOAD DATASET
# ===============================
path = "data/dataset_finale_Stoichiometry_CLEAN.csv"
df = pd.read_csv(path, sep=";")

# ===============================
# 2. NORMALIZE COFACTOR COLUMN
# ===============================
df["Cofactor"] = (
    df["Cofactor"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
)

# ===============================
# 3. AGGREGATE BY REACTION
# ===============================
reazioni = (
    df.groupby("Rhea_ID")
    .agg(
        has_cofactor=("Cofactor", lambda x: x.notna().any())
    )
    .reset_index()
)

# ===============================
# 4. FINAL COUNTS
# ===============================
num_con_cofattore = reazioni["has_cofactor"].sum()
num_senza_cofattore = (~reazioni["has_cofactor"]).sum()
totale_reazioni = len(reazioni)

print(f"Total unique reactions: {totale_reazioni}")
print(f"Reactions WITH cofactors: {num_con_cofattore}")
print(f"Reactions WITHOUT cofactors: {num_senza_cofattore}")


In [ ]:

# =====================================================
# STATISTICAL ANALYSIS OF THE FINAL DATASET
# =====================================================

import pandas as pd

# =========================
# 1. LOAD DATASET
# =========================
file_path = "data/dataset_finale_Stoichiometry_CLEAN.csv"

df = pd.read_csv(file_path, sep=";", encoding="utf-8")

print("Dataset loaded successfully")
print(f"Total number of rows: {df.shape[0]}")
print(f"Total number of columns: {df.shape[1]}")
print("-" * 50)

# =========================
# 2. BASIC STATISTICS
# =========================

# Unique diseases (ORPHAcode is the most reliable identifier)
unique_diseases = df["ORPHAcode"].nunique()

# Enzyme-coding genes
unique_genes = df["Gene"].dropna().nunique()

# Unique enzymatic reactions (Rhea_ID)
unique_reactions = df["Rhea_ID"].dropna().nunique()

# Disease–enzyme associations
# (unique ORPHAcode + Gene pairs)
disease_enzyme_associations = (
    df[["ORPHAcode", "Gene"]]
    .dropna()
    .drop_duplicates()
    .shape[0]
)

# =========================
# 3. PRINT RESULTS
# =========================
print("📊 STATISTICAL RESULTS")
print("-" * 50)
print(f"Unique diseases (ORPHAcode): {unique_diseases}")
print(f"Unique enzyme-coding genes: {unique_genes}")
print(f"Unique enzymatic reactions (Rhea_ID): {unique_reactions}")
print(f"Unique disease–enzyme associations: {disease_enzyme_associations}")

# =========================
# 4. ADDITIONAL STATISTICAL ANALYSIS
# =========================

# Average number of reactions per disease
reactions_per_disease = (
    df[["ORPHAcode", "Rhea_ID"]]
    .dropna()
    .drop_duplicates()
    .groupby("ORPHAcode")
    .size()
)

print("-" * 50)
print(f"Mean reactions per disease: {reactions_per_disease.mean():.2f}")
print(f"Median reactions per disease: {reactions_per_disease.median():.2f}")

# Top 10 diseases with the highest number of reactions
top_diseases = reactions_per_disease.sort_values(ascending=False).head(10)

print("\nTop 10 diseases by number of reactions:")
print(top_diseases)


In [ ]:

# =====================================================
# STATISTICAL ANALYSIS OF PHENOTYPE AND COFACTOR ANNOTATIONS
# =====================================================

import pandas as pd

# =========================
# 1. LOAD DATASET
# =========================
file_path = "data/dataset_finale_Stoichiometry_CLEAN.csv"
df = pd.read_csv(file_path, sep=";", encoding="utf-8")

# =========================
# 2. DEFINE BOOLEAN FLAGS
# =========================
df["has_phenotype"] = df["HPO_ID"].notna()
df["has_cofactor"] = df["Cofactor"].notna()

# =========================
# 3. COUNTING FUNCTIONS
# =========================
def count_unique(df, column):
    return df[column].dropna().nunique()

def count_condition(df, column, condition):
    return df.loc[condition, column].dropna().nunique()

# =========================
# 4. PHENOTYPE ANALYSIS
# =========================
phenotype_results = {
    "Diseases_with_phenotype": count_condition(df, "ORPHAcode", df["has_phenotype"]),
    "Diseases_without_phenotype": count_condition(df, "ORPHAcode", ~df["has_phenotype"]),
    "Genes_with_phenotype": count_condition(df, "Gene", df["has_phenotype"]),
    "Genes_without_phenotype": count_condition(df, "Gene", ~df["has_phenotype"]),
    "Reactions_with_phenotype": count_condition(df, "Rhea_ID", df["has_phenotype"]),
    "Reactions_without_phenotype": count_condition(df, "Rhea_ID", ~df["has_phenotype"]),
}

# =========================
# 5. COFACTOR ANALYSIS
# =========================
cofactor_results = {
    "Diseases_with_cofactor": count_condition(df, "ORPHAcode", df["has_cofactor"]),
    "Diseases_without_cofactor": count_condition(df, "ORPHAcode", ~df["has_cofactor"]),
    "Genes_with_cofactor": count_condition(df, "Gene", df["has_cofactor"]),
    "Genes_without_cofactor": count_condition(df, "Gene", ~df["has_cofactor"]),
    "Reactions_with_cofactor": count_condition(df, "Rhea_ID", df["has_cofactor"]),
    "Reactions_without_cofactor": count_condition(df, "Rhea_ID", ~df["has_cofactor"]),
}

# =========================
# 6. COMBINED PHENOTYPE AND COFACTOR ANALYSIS
# =========================
combined_results = {
    "Diseases_with_pheno_and_cofactor": count_condition(
        df, "ORPHAcode", df["has_phenotype"] & df["has_cofactor"]
    ),
    "Diseases_with_pheno_no_cofactor": count_condition(
        df, "ORPHAcode", df["has_phenotype"] & ~df["has_cofactor"]
    ),
    "Diseases_no_pheno_with_cofactor": count_condition(
        df, "ORPHAcode", ~df["has_phenotype"] & df["has_cofactor"]
    ),
    "Diseases_no_pheno_no_cofactor": count_condition(
        df, "ORPHAcode", ~df["has_phenotype"] & ~df["has_cofactor"]
    ),

    "Genes_with_pheno_and_cofactor": count_condition(
        df, "Gene", df["has_phenotype"] & df["has_cofactor"]
    ),
    "Genes_with_pheno_no_cofactor": count_condition(
        df, "Gene", df["has_phenotype"] & ~df["has_cofactor"]
    ),
    "Genes_no_pheno_with_cofactor": count_condition(
        df, "Gene", ~df["has_phenotype"] & df["has_cofactor"]
    ),
    "Genes_no_pheno_no_cofactor": count_condition(
        df, "Gene", ~df["has_phenotype"] & ~df["has_cofactor"]
    ),

    "Reactions_with_pheno_and_cofactor": count_condition(
        df, "Rhea_ID", df["has_phenotype"] & df["has_cofactor"]
    ),
    "Reactions_with_pheno_no_cofactor": count_condition(
        df, "Rhea_ID", df["has_phenotype"] & ~df["has_cofactor"]
    ),
    "Reactions_no_pheno_with_cofactor": count_condition(
        df, "Rhea_ID", ~df["has_phenotype"] & df["has_cofactor"]
    ),
    "Reactions_no_pheno_no_cofactor": count_condition(
        df, "Rhea_ID", ~df["has_phenotype"] & ~df["has_cofactor"]
    ),
}

# =========================
# 7. PRINT RESULTS
# =========================
print("\n📌 PHENOTYPE")
for k, v in phenotype_results.items():
    print(f"{k}: {v}")

print("\n📌 COFACTOR")
for k, v in cofactor_results.items():
    print(f"{k}: {v}")

print("\n📌 PHENOTYPE + COFACTOR")
for k, v in combined_results.items():
    print(f"{k}: {v}")


In [ ]:

# =====================================================
# VISUALIZATION OF PHENOTYPE AND COFACTOR ANNOTATION COVERAGE
# =====================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# 1. LOAD DATASET
# =========================
file_path = "data/dataset_finale_Stoichiometry_CLEAN.csv"
df = pd.read_csv(file_path, sep=";", encoding="utf-8")

# =========================
# 2. DEFINE BOOLEAN FLAGS
# =========================
df["has_phenotype"] = df["HPO_ID"].notna()
df["has_cofactor"] = df["Cofactor"].notna()

# =========================
# 3. AGGREGATED COUNTS
# =========================
summary = pd.DataFrame({
    "Entity": ["Diseases", "Genes", "Reactions"],
    "With phenotype": [
        df.loc[df["has_phenotype"], "ORPHAcode"].nunique(),
        df.loc[df["has_phenotype"], "Gene"].nunique(),
        df.loc[df["has_phenotype"], "Rhea_ID"].nunique(),
    ],
    "Without phenotype": [
        df.loc[~df["has_phenotype"], "ORPHAcode"].nunique(),
        df.loc[~df["has_phenotype"], "Gene"].nunique(),
        df.loc[~df["has_phenotype"], "Rhea_ID"].nunique(),
    ],
    "With cofactor": [
        df.loc[df["has_cofactor"], "ORPHAcode"].nunique(),
        df.loc[df["has_cofactor"], "Gene"].nunique(),
        df.loc[df["has_cofactor"], "Rhea_ID"].nunique(),
    ],
    "Without cofactor": [
        df.loc[~df["has_cofactor"], "ORPHAcode"].nunique(),
        df.loc[~df["has_cofactor"], "Gene"].nunique(),
        df.loc[~df["has_cofactor"], "Rhea_ID"].nunique(),
    ]
})

# =========================
# 4. BAR PLOT – ANNOTATION COVERAGE
# =========================
summary_melted = summary.melt(
    id_vars="Entity",
    value_vars=["With phenotype", "Without phenotype", "With cofactor", "Without cofactor"],
    var_name="Annotation",
    value_name="Count"
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=summary_melted,
    x="Entity",
    y="Count",
    hue="Annotation"
)
plt.title("Coverage of Phenotype and Cofactor Annotations")
plt.ylabel("Unique count")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()

# =========================
# 5. STACKED BAR PLOT – PHENOTYPE × COFACTOR (REACTIONS)
# =========================
reaction_classes = pd.DataFrame({
    "Phenotype + Cofactor": df.loc[df["has_phenotype"] & df["has_cofactor"], "Rhea_ID"].nunique(),
    "Phenotype only": df.loc[df["has_phenotype"] & ~df["has_cofactor"], "Rhea_ID"].nunique(),
    "Cofactor only": df.loc[~df["has_phenotype"] & df["has_cofactor"], "Rhea_ID"].nunique(),
    "Neither": df.loc[~df["has_phenotype"] & ~df["has_cofactor"], "Rhea_ID"].nunique()
}, index=["Reactions"])

reaction_classes.T.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 6)
)

plt.title("Reaction Annotation Space: Phenotype vs Cofactor")
plt.ylabel("Unique reactions")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()


In [ ]:

# =====================================================
# DISEASE COMPLEXITY BASED ON THE NUMBER OF REACTIONS
# =====================================================

# Define reaction count intervals and labels
bins = [0, 1, 3, 5, 10, 100]
labels = ["1", "2–3", "4–5", "6–10", ">10"]

# Classify diseases by number of associated reactions
reaction_classes = pd.cut(reactions_per_disease, bins=bins, labels=labels)
class_counts = reaction_classes.value_counts().sort_index()

# =====================================================
# VISUALIZE DISEASE COMPLEXITY DISTRIBUTION
# =====================================================

plt.figure(figsize=(7, 5))
class_counts.plot(kind="bar")
plt.title("Disease Complexity Based on Number of Reactions")
plt.ylabel("Number of diseases")
plt.xlabel("Reactions per disease")
plt.tight_layout()
plt.show()

# Compare the number of diseases associated with
# few reactions versus those associated with many reactions.


In [ ]:

# =====================================================
# DISEASE COMPLEXITY ANALYSIS – PIE CHART
# =====================================================

import pandas as pd
import matplotlib.pyplot as plt

# =========================
# LOAD DATASET
# =========================
file_path = "data/dataset_finale_Stoichiometry_CLEAN.csv"
df = pd.read_csv(file_path, sep=";", encoding="utf-8")

# =========================
# CALCULATE REACTIONS PER DISEASE
# =========================
reactions_per_disease = (
    df[["ORPHAcode", "Rhea_ID"]]
    .dropna()
    .drop_duplicates()
    .groupby("ORPHAcode")
    .size()
)

# =========================
# DEFINE DISEASE COMPLEXITY CLASSES
# =========================
def classify(n):
    if n == 1:
        return "1 reaction"
    elif 2 <= n <= 3:
        return "2–3 reactions"
    elif 4 <= n <= 5:
        return "4–5 reactions"
    else:
        return ">5 reactions"

complexity_classes = reactions_per_disease.apply(classify)
class_counts = complexity_classes.value_counts().sort_index()

# =========================
# PIE CHART
# =========================
plt.figure(figsize=(7, 7))
plt.pie(
    class_counts,
    labels=class_counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Disease Complexity Based on Number of Reactions")
plt.tight_layout()
plt.show()

# Most diseases are associated with a small number of
# biochemical reactions, while a minority involves
# multiple reactions, highlighting a strongly skewed
# complexity landscape.


In [ ]:

# =====================================================
# BINARY CLASSIFICATION OF METABOLIC DISEASE COMPLEXITY
# =====================================================

# =========================
# CLASSIFY DISEASES AS SIMPLE OR COMPLEX
# =========================
# Simple: 1–2 associated reactions
# Complex: more than 2 associated reactions

binary_class = reactions_per_disease.apply(
    lambda x: "Simple (≤2 reactions)" if x <= 2 else "Complex (>2 reactions)"
)

# =========================
# COUNT DISEASES IN EACH CATEGORY
# =========================
binary_counts = binary_class.value_counts()

# =========================
# PIE CHART – SIMPLE VS COMPLEX DISEASES
# =========================
plt.figure(figsize=(6, 6))
plt.pie(
    binary_counts,
    labels=binary_counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Simple vs Complex Metabolic Diseases")
plt.tight_layout()
plt.show()
